In [68]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np


In [69]:
df=pd.read_csv("collegePlace.csv")

In [70]:
df.head()
df.columns.value_counts()

Age                  1
Gender               1
Stream               1
Internships          1
CGPA                 1
Hostel               1
HistoryOfBacklogs    1
PlacedOrNot          1
Name: count, dtype: int64

In [71]:
#bool_cols = df.select_dtypes(include='bool').columns

#df[bool_cols] = df[bool_cols].astype(int)
#from sklearn.preprocessing import LabelEncoder,StandardScaler

#le=LabelEncoder()
#df["Gender"]=le.fit_transform(df["Gender"])
#df["Stream"]=le.fit_transform(df["Stream"])
# One-hot encode categorical features
df = pd.get_dummies(df, columns=["Stream", "Gender"], drop_first=True)

# Convert the resulting boolean columns to integers (0 or 1)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [72]:
df.dtypes


Age                                     int64
Internships                             int64
CGPA                                    int64
Hostel                                  int64
HistoryOfBacklogs                       int64
PlacedOrNot                             int64
Stream_Computer Science                 int64
Stream_Electrical                       int64
Stream_Electronics And Communication    int64
Stream_Information Technology           int64
Stream_Mechanical                       int64
Gender_Male                             int64
dtype: object

In [73]:
df.head()
x=df.drop("PlacedOrNot",axis=1)
y=df["PlacedOrNot"]


In [74]:
#train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [75]:
#scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)


In [76]:
#tensors
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.long)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.long)

In [77]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [78]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle="True")
test_loader = DataLoader(test_dataset,batch_size=32)

In [79]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x.shape[1],64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,2)
            
        )
    def forward (self,x):
        return self.model(x)

In [80]:
model=ANN()
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.0005)

In [81]:
epochs=200
for epoch in range (epochs):
    model.train()
    running_loss=0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/200, loss = 0.6463207650184631
epoch = 2/200, loss = 0.4988900055487951
epoch = 3/200, loss = 0.44611693223317467
epoch = 4/200, loss = 0.4048976856470108
epoch = 5/200, loss = 0.38832947810490925
epoch = 6/200, loss = 0.3690961366891861
epoch = 7/200, loss = 0.35626594722270966
epoch = 8/200, loss = 0.3425007172425588
epoch = 9/200, loss = 0.34441779534022016
epoch = 10/200, loss = 0.34119377811749774
epoch = 11/200, loss = 0.33067461609840393
epoch = 12/200, loss = 0.32952661434809366
epoch = 13/200, loss = 0.32415301611026126
epoch = 14/200, loss = 0.32319303770860036
epoch = 15/200, loss = 0.3330498278141022
epoch = 16/200, loss = 0.3191395914554596
epoch = 17/200, loss = 0.3107009303569794
epoch = 18/200, loss = 0.3142089136441549
epoch = 19/200, loss = 0.30903574605782824
epoch = 20/200, loss = 0.30951993624369306
epoch = 21/200, loss = 0.306624626715978
epoch = 22/200, loss = 0.307754174520572
epoch = 23/200, loss = 0.3021717576185862
epoch = 24/200, loss = 0.301064554

In [82]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(yb.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Accuracy:", accuracy_score(y_true, y_pred) * 100)

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Accuracy: 88.88888888888889

Confusion Matrix
[[269  11]
 [ 55 259]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.96      0.89       280
           1       0.96      0.82      0.89       314

    accuracy                           0.89       594
   macro avg       0.89      0.89      0.89       594
weighted avg       0.90      0.89      0.89       594



In [83]:
model.eval()

test_loss = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        loss = criteria(outputs, yb)
        test_loss += loss.item()

print("Test Loss:", test_loss / len(test_loader))

Test Loss: 0.2919243197692068


In [84]:
import pickle

# 1. Save the PyTorch Model Weights
torch.save(model.state_dict(), "placement_model.pth")

# 2. Save the StandardScaler instance
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and Scaler saved successfully!")

Model and Scaler saved successfully!
